# 02 · Resonators (contest exercise)

**Style C: contest simulation**, modelled on T24-O-HYPER ("Lost in Hyperspace": frozen
`LinearRegression`, feature cap, scaled RMSE, 5 min CPU) and T24-H-ML ("Pulse of the Machine": multichannel
signal arrays, "the data itself was corrupted"). **Traces to** `notes/ioai-task-patterns.md` P1, P2, P5, P16.

> **Reference solution.** A robust spectrum gives frequency (with a parabolic peak fit), a log-envelope slope gives
> damping, and the envelope intercept gives amplitude. Spikes are clipped with a MAD rule and dead channels are
> skipped. The fixed `LinearRegression` then only needs to calibrate these physical estimates.

## 1. Problem Description

A factory tests tuning forks ("resonators") by tapping them and recording the ring-down with **6 microphones**.
From the recording it needs three properties of each fork: its **frequency** (Hz), its **strike amplitude**
and its **damping rate** (1/s). The production line only runs a certified linear regressor, so your whole
contribution is the feature extractor.

## 2. Dataset

Each sample is a float32 array of shape `(6, 128)`: 6 microphones × 128 time steps sampled at **64 Hz**
(2 seconds).
- The order of the microphones is **random** for every sample, and each microphone has its own gain.
- Some microphones fail: they are silent, or they pick up electrical spikes.

| Split | Samples | Targets |
|---|---|---|
| `X_train`, `Y_train` | 4,000 | `Y[:, 0]` frequency, `Y[:, 1]` amplitude, `Y[:, 2]` damping |
| `X_test_a` | 2,000 | hidden (Leaderboard A) |
| `X_test_b` | 2,000 | hidden (Leaderboard B, **final**) |

## 3. Task

Write `feature_extractor(X) -> F`, mapping `(n, 6, 128)` to `(n, k)` with **k ≤ 20**. The **frozen** scorer
fits one `LinearRegression()` (default settings) per target on your training features and predicts the test
features.

## 4. Submission

`submission.zip` containing `predictions_a.csv` and `predictions_b.csv`, each with a header
`ID,frequency,amplitude,damping` and one row per test sample in order. The provided cell writes them for you.
Keep the fitting code as it is.

## 5. Scoring

For each target, the RMSE divided by the standard deviation of that target on the test split (normalised
RMSE). The score is the **mean of the three normalised RMSEs, and lower is better**. Predicting the mean scores 1.0.

- Baseline (first 20 raw values): **1.00**
- Reference solution: **0.12**, which is your target (≤ 0.12)

## 6. Requirements (copied from T24-O-HYPER)

- `feature_extractor` + fitting + prediction must run in **≤ 3 minutes on CPU**. Exploration does not count.
- Only `LinearRegression()` with default hyperparameters is allowed as the predictor. Supervised models may
  not be used as feature extractors; unsupervised ones (PCA, clustering, autoencoders) may.
- No pretrained models and no external data. You may not use the targets inside `feature_extractor`.
- Do not read the generator blob. Treat it as the organisers' server.

In [ ]:
import time, os, zipfile, base64, zlib, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
seed = 42
random.seed(seed); np.random.seed(seed)
FS = 64

### Data loading (DO NOT CHANGE, DO NOT DECODE)

In [ ]:
_BLOB = ("eNp1U0tvozAQvudXzNFsvS7Qh1ZpOUSR9thTDqlQVHlhACtgLBu023/fsR3RbJpyADTfY/wNgxrMaCfQ82DeQTrQZlVjA2+DPCLTHBxinaxXQJfVLRREEFbqehwE8eTcT29UZ4EWWI3jsOOwJerjPYcs/8XhMSBTVEuSt8h2CdwSOWoIIRcxa9WMdmA5hzzloJMn2FxAqXjgcBex9gvmhSIog28rlXbXDDJ/8/G2ifcJtNsivogBpWYZhyOiqdXgip2dMRqaTjq8NKSe8MMnM2oxDew9MdmmXHN4GTXG+4GooY0vrz9rJMd/hv1sv9Kn6BauQHRKs6UlPZsrGriJhz1vc3asPeE+hKYEsvcZUpHRTPbCddKc0v6R9Slr/OJMJ/BMxPwhwFV3QpWesEXrvM12GX6N38ijmmYHCpT2Kf52aJFRu6RMD+slrWqCSanOaiFASYOuOqofqEEq0gXF3uH/3OPpCFU3qoq2jpaRg0XTywqL35L4yTfetAAHuCkuxuS3OSpe4zq7SVZHVjYcNhxaUmURtjjNVtNApZveDf1LRjT9KKe7POHweqW6+gA4pvaw")
exec(zlib.decompress(base64.b64decode(_BLOB)).decode())      # defines the organisers' generator
X_train, Y_train = _make(4000, 11)
X_test_a = _make(2000, 22)[0]
X_test_b = _make(2000, 33)[0]
TARGETS = ["frequency", "amplitude", "damping"]
print(X_train.shape, Y_train.shape, X_test_a.shape, X_test_b.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
t = np.arange(128) / FS
for ax, i in zip(axes, range(3)):
    ax.plot(t, X_train[i].T, lw=0.8)
    ax.set_title(f"f={Y_train[i,0]:.1f} Hz  A={Y_train[i,1]:.2f}  γ={Y_train[i,2]:.2f}")
plt.tight_layout(); plt.show()

### The frozen scorer (DO NOT CHANGE)

In [ ]:
def fit_predict(F_train, Y_train, F_test):
    assert F_train.ndim == 2 and F_train.shape[1] <= 20, f"at most 20 features, got {F_train.shape}"
    assert np.isfinite(F_train).all() and np.isfinite(F_test).all(), "features contain NaN/inf"
    return np.column_stack([LinearRegression().fit(F_train, Y_train[:, k]).predict(F_test) for k in range(3)])

def nrmse(Y_true, Y_pred):
    per = np.sqrt(((Y_true - Y_pred) ** 2).mean(0)) / Y_true.std(0)
    return per.mean(), per

### Reference feature extractor

The generator (revealed after the contest) is $A\,g_c\,e^{-\gamma t}\sin(2\pi f t + \phi_c) + \text{noise}$,
with per-microphone gains $g_c$ whose mean is 1. You can recover that form by looking at the plots:

1. **Spikes**: clip each channel at median ± 6·MAD. **Dead channels** have std ≈ 0, so mask them out.
2. **Frequency**: sum the power spectra over live channels (phase-free and order-free), take the argmax bin and
   refine it with a parabola through the peak and its neighbours.
3. **Damping and amplitude**: the Hilbert envelope is ≈ $A g_c e^{-\gamma t}$, so a straight-line fit of
   $\log(\text{envelope})$ against $t$ gives slope $-\gamma$ and intercept $\log(A g_c)$. Take the median slope
   over channels, and the mean of $e^{\text{intercept}}$ (the gains average to 1).
4. Add a few products and powers so the *linear* model can correct the estimators' bias.

In [ ]:
from scipy.signal import hilbert

def feature_extractor(X):
    X = X.astype(np.float64)
    T = X.shape[2]; t = np.arange(T) / FS
    med = np.median(X, axis=2, keepdims=True)
    mad = np.median(np.abs(X - med), axis=2, keepdims=True) * 1.4826 + 1e-6
    Xc = np.clip(X, med - 6 * mad, med + 6 * mad)                     # spikes
    alive = Xc.std(axis=2) > 1e-3                                       # dead microphones
    P = (np.abs(np.fft.rfft(Xc, axis=2)) ** 2 * alive[:, :, None]).sum(1)
    k = np.clip(P[:, 1:].argmax(1) + 1, 1, P.shape[1] - 2)
    i = np.arange(len(P))
    a, b, c = P[i, k - 1], P[i, k], P[i, k + 1]
    f_hat = (k + 0.5 * (a - c) / (a - 2 * b + c + 1e-12)) * FS / T
    env = np.abs(hilbert(Xc, axis=2))
    env = np.apply_along_axis(lambda v: np.convolve(v, np.ones(9) / 9, mode="same"), 2, env)
    tt = t[8:96]; L = np.log(env[:, :, 8:96] + 0.05); tm = tt - tt.mean()
    slope = (L * tm).sum(2) / (tm ** 2).sum()
    inter = L.mean(2) - slope * tt.mean()
    slope = np.where(alive, slope, np.nan); inter = np.where(alive, inter, np.nan)
    g_hat = -np.nanmedian(slope, 1)
    A_mean, A_med = np.nanmean(np.exp(inter), 1), np.nanmedian(np.exp(inter), 1)
    rms = np.nanmean(np.where(alive, np.sqrt((Xc ** 2).mean(2)), np.nan), 1)
    F = np.column_stack([f_hat, f_hat ** 2, g_hat, g_hat ** 2, g_hat ** 3, A_mean, A_med, A_mean ** 2,
                         np.log(A_mean), A_mean * g_hat, rms, rms * g_hat, g_hat * f_hat])
    return np.nan_to_num(F)

### Local validation (hold out part of the training set)

In [ ]:
cut = 3000
t0 = time.time()
F_all = feature_extractor(X_train)
score, per = nrmse(Y_train[cut:], fit_predict(F_all[:cut], Y_train[:cut], F_all[cut:]))
print(f"local nRMSE {score:.4f}  per target {dict(zip(TARGETS, per.round(4)))}  ({time.time() - t0:.1f}s)")

### Submission

In [ ]:
t0 = time.time()
F_train, F_a, F_b = feature_extractor(X_train), feature_extractor(X_test_a), feature_extractor(X_test_b)
pred_a, pred_b = fit_predict(F_train, Y_train, F_a), fit_predict(F_train, Y_train, F_b)
graded_seconds = time.time() - t0
for name, P in [("predictions_a.csv", pred_a), ("predictions_b.csv", pred_b)]:
    df = pd.DataFrame(P, columns=TARGETS); df.insert(0, "ID", np.arange(len(P)))
    df.to_csv(name, index=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("predictions_a.csv"); z.write("predictions_b.csv")
print(f"graded part took {graded_seconds:.1f}s (limit 180s)")
assert graded_seconds < 180, "too slow: the platform would reject this run"

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    exec(zlib.decompress(base64.b64decode(_BLOB)).decode(), globals())
    truth = {"predictions_a.csv": _make(2000, 22)[1], "predictions_b.csv": _make(2000, 33)[1]}
    res = {}
    with zipfile.ZipFile(zip_path) as z:
        for name, Y in truth.items():
            try:
                df = pd.read_csv(z.open(name))
                P = df[TARGETS].to_numpy(dtype=float)
                assert len(P) == len(Y) and (df["ID"].to_numpy() == np.arange(len(Y))).all()
                res[name] = nrmse(Y, P)[0]
            except Exception as e:
                print(name, "rejected:", e); res[name] = float("inf")
    return res["predictions_a.csv"], res["predictions_b.csv"]

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}   (lower is better)")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")